In [1]:
!pip install prophet xgboost fastapi uvicorn pyngrok holidays openpyxl statsmodels tensorflow

In [2]:
from google.colab import files

uploaded = files.upload()

Saving Forecasting Case- Study.xlsx to Forecasting Case- Study.xlsx


In [3]:
import pandas as pd

df = pd.read_excel("Forecasting Case- Study.xlsx")

df.head()

,State,Date,Total,Category
0,Alabama,2019-01-12 00:00:00,109574036.0,Beverages
1,Arizona,2019-01-12 00:00:00,109101594.6,Beverages
2,Arkansas,2019-01-12 00:00:00,58049432.2,Beverages
3,California,2019-01-12 00:00:00,444766890.6,Beverages
4,Colorado,2019-01-12 00:00:00,89816716.3,Beverages


In [4]:
df['Date'] = pd.to_datetime(df['Date'])

In [5]:
df = df.sort_values(['State', 'Date'])

In [6]:
print(df.columns.tolist())
print(df.head())

['State', 'Date', 'Total', 'Category']
       State       Date        Total   Category
0    Alabama 2019-01-12  109574036.0  Beverages
43   Alabama 2019-03-11  112189103.8  Beverages
86   Alabama 2019-06-10  129106730.4  Beverages
129  Alabama 2019-08-12  108083723.8  Beverages
172  Alabama 2019-10-11  110932912.8  Beverages


In [7]:
df.columns = df.columns.str.strip()

In [8]:
print(df.columns)

Index(['State', 'Date', 'Total', 'Category'], dtype='object')


In [9]:
import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_error
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

# Load data
df = pd.read_excel("Forecasting Case- Study.xlsx")

# Clean columns
df.columns = df.columns.str.strip()

# Rename 'Total' to 'Sales' for consistency with feature engineering and modeling
df = df.rename(columns={"Total": "Sales"})

# Convert Date
df['Date'] = pd.to_datetime(df['Date'])

# Sort
df = df.sort_values(['State', 'Date'])

# -----------------------------
# Feature Engineering
# -----------------------------
def create_features(data):
    data['dayofweek'] = data['Date'].dt.dayofweek
    data['month'] = data['Date'].dt.month
    data['week'] = data['Date'].dt.isocalendar().week.astype(int)

    data['lag_1'] = data['Sales'].shift(1)
    data['lag_7'] = data['Sales'].shift(7)
    data['rolling_mean_7'] = data['Sales'].rolling(7).mean()

    return data

df = df.groupby('State', group_keys=False).apply(create_features).reset_index(drop=True)

# One-hot encode the 'Category' column before dropping NaNs
df = pd.get_dummies(df, columns=['Category'], drop_first=True, dtype=int)

df = df.dropna()

# -----------------------------
# Model Training per State
# -----------------------------
models = {}

# Dynamically get feature columns after one-hot encoding
# Exclude 'Sales', 'State', 'Date' and the original 'Category' which is now replaced by dummies
feature_columns = [col for col in df.columns if col not in ['Sales', 'State', 'Date', 'Category']]

for state in df['State'].unique():
    state_df = df[df['State'] == state]

    train = state_df[:-8]
    test = state_df[-8:]

    X_train = train[feature_columns]
    y_train = train['Sales']

    X_test = test[feature_columns]
    y_test = test['Sales']

    rf = RandomForestRegressor()
    xgb = XGBRegressor()

    rf.fit(X_train, y_train)
    xgb.fit(X_train, y_train)

    rf_pred = rf.predict(X_test)
    xgb_pred = xgb.predict(X_test)

    rf_mae = mean_absolute_error(y_test, rf_pred)
    xgb_mae = mean_absolute_error(y_test, xgb_pred)

    best_model = rf if rf_mae < xgb_mae else xgb

    models[state] = best_model

    print(f"{state} -> RF MAE: {rf_mae}, XGB MAE: {xgb_mae}")

# -----------------------------
# Forecast Next 8 Weeks
# -----------------------------
forecast_results = []

for state in df['State'].unique():
    state_df = df[df['State'] == state].copy()

    model = models[state]

    last_data = state_df.tail(8)

    X_future = last_data[feature_columns]

    preds = model.predict(X_future)

    temp = last_data[['Date']].copy()
    temp['State'] = state
    temp['Forecast'] = preds

    forecast_results.append(temp)

forecast_df = pd.concat(forecast_results)

print("\nFinal Forecast:")
print(forecast_df)

/tmp/ipykernel_1310/1249498450.py:36: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df = df.groupby('State', group_keys=False).apply(create_features).reset_index(drop=True)


Alabama -> RF MAE: 10612887.279500026, XGB MAE: 10624224.475000005
Arizona -> RF MAE: 10424698.891875088, XGB MAE: 11141121.362500004
Arkansas -> RF MAE: 5369244.377374975, XGB MAE: 4218674.612499999
California -> RF MAE: 41306151.717124835, XGB MAE: 38971477.44999996
Colorado -> RF MAE: 6726216.586875003, XGB MAE: 4828588.187499996
Connecticut -> RF MAE: 3627249.641124999, XGB MAE: 2835101.7749999966
Florida -> RF MAE: 38580860.58225009, XGB MAE: 44458001.774999976
Georgia -> RF MAE: 15297095.000249974, XGB MAE: 17056762.11249999
Illinois -> RF MAE: 16865813.761250027, XGB MAE: 17998447.125000007
Indiana -> RF MAE: 8710128.136624958, XGB MAE: 7337014.7749999985
Iowa -> RF MAE: 6583782.476249997, XGB MAE: 7684469.262500003
Kansas -> RF MAE: 4766822.771499993, XGB MAE: 4559979.187500004
Kentucky -> RF MAE: 7101525.421124965, XGB MAE: 6246201.77500001
Louisiana -> RF MAE: 7945039.87224995, XGB MAE: 7243582.799999997
Maine -> RF MAE: 3264096.067750009, XGB MAE: 4254856.0125
Maryland -> RF

In [10]:
from fastapi import FastAPI
import pandas as pd

app = FastAPI()

@app.get("/forecast/{state}")
def get_forecast(state: str):
    result = forecast_df[forecast_df['State'] == state]
    return result.to_dict(orient='records')